# Benchmark de escalabilidad — Pipeline EDA REES46

In [1]:

from pathlib import Path
import sys
import time
import gc
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.pipeline import run_multidimensional_pipeline

DATA_PATH = PROJECT_DIR / "data" / "raw" / "2019-Oct.csv"

# Cargar una única muestra de 100K eventos
df = pd.read_csv(
    DATA_PATH,
    nrows=100_000
)

df["event_id"] = np.arange(len(df))
df["event_timestamp"] = pd.to_datetime(
    df["event_time"],
    utc=True
)

print("=== DATASET DE ESCALABILIDAD ===")
print("Eventos disponibles:", len(df))
print("Columnas:", len(df.columns))
print("Memoria DataFrame (MiB):",
      round(df.memory_usage(deep=True).sum() / 1024**2, 2))


=== DATASET DE ESCALABILIDAD ===
Eventos disponibles: 100000
Columnas: 11
Memoria DataFrame (MiB): 16.38


In [2]:

import os
import psutil

SIZES = [1_000, 10_000, 100_000]
process = psutil.Process(os.getpid())

results = []

for n in SIZES:
    print(f"\n=== PROCESANDO {n:,} EVENTOS ===")

    gc.collect()
    memory_before = process.memory_info().rss

    # Preparar eventos fuera de la medición
    events = df.head(n).to_dict("records")

    start = time.perf_counter()

    result = run_multidimensional_pipeline(
        events=events,
        batch_size=1000,
        cache_capacity=100,
        max_keys=32,
        leaf_size=40,
        dedup_method="set"
    )

    elapsed = time.perf_counter() - start
    memory_after = process.memory_info().rss

    metrics = result["metrics"]

    results.append({
        "eventos": n,
        "tiempo_s": elapsed,
        "throughput_eventos_s": n / elapsed,
        "rss_delta_mib": (
            memory_after - memory_before
        ) / 1024**2,
        "productos": metrics["catalog_products"],
        "csr_nnz": metrics["csr_nnz"],
        "kdtree_productos": metrics["kdtree_products"]
    })

    print(f"Tiempo: {elapsed:.3f} s")
    print(f"Throughput: {n / elapsed:,.0f} eventos/s")
    print("Productos:", metrics["catalog_products"])
    print("KD-Tree:", metrics["kdtree_products"])

    # Liberar referencias para la siguiente ejecución
    del result, events
    gc.collect()

df_pipeline_scalability = pd.DataFrame(results)

print("\n=== RESUMEN DE ESCALABILIDAD ===")
display(df_pipeline_scalability.round(3))



=== PROCESANDO 1,000 EVENTOS ===
Tiempo: 0.030 s
Throughput: 32,936 eventos/s
Productos: 534
KD-Tree: 534

=== PROCESANDO 10,000 EVENTOS ===
Tiempo: 0.115 s
Throughput: 87,255 eventos/s
Productos: 4148
KD-Tree: 4148

=== PROCESANDO 100,000 EVENTOS ===
Tiempo: 0.849 s
Throughput: 117,737 eventos/s
Productos: 20621
KD-Tree: 20621

=== RESUMEN DE ESCALABILIDAD ===


,eventos,tiempo_s,throughput_eventos_s,rss_delta_mib,productos,csr_nnz,kdtree_productos
0,1000,0.030,32936.015,1.117,534,701,534
1,10000,0.115,87255.010,12.160,4148,7026,4148
2,100000,0.849,117737.273,133.152,20621,65919,20621


## 3. Diagnóstico de memoria del pipeline

In [3]:

import gc
import time
import tracemalloc
import pandas as pd

# Comenzamos con 10K para verificar el diagnóstico
N = 10_000

events_test = df.head(N).to_dict("records")

gc.collect()
tracemalloc.start()

start = time.perf_counter()

result_memory = run_multidimensional_pipeline(
    events=events_test,
    batch_size=1000,
    cache_capacity=100,
    max_keys=32,
    leaf_size=40,
    dedup_method="set"
)

elapsed = time.perf_counter() - start

current, peak = tracemalloc.get_traced_memory()

print("=== DIAGNÓSTICO DE MEMORIA ===")
print(f"Eventos procesados: {N:,}")
print(f"Tiempo: {elapsed:.3f} s")
print(f"Memoria Python actual: {current / 1024**2:.2f} MiB")
print(f"Pico rastreado: {peak / 1024**2:.2f} MiB")

print("\n=== PRINCIPALES ASIGNACIONES ===")

snapshot = tracemalloc.take_snapshot()

for stat in snapshot.statistics("filename")[:10]:
    print(stat)

tracemalloc.stop()


=== DIAGNÓSTICO DE MEMORIA ===
Eventos procesados: 10,000
Tiempo: 0.455 s
Memoria Python actual: 3.63 MiB
Pico rastreado: 4.27 MiB

=== PRINCIPALES ASIGNACIONES ===
c:\Users\bryan\Proyecto_EDA_UTEC\src\storage.py:0: size=1282 KiB, count=12596, average=104 B
c:\Users\bryan\Proyecto_EDA_UTEC\src\sparse_matrix.py:0: size=631 KiB, count=8581, average=75 B
c:\Users\bryan\Proyecto_EDA_UTEC\src\multidimensional_index.py:0: size=461 KiB, count=8076, average=58 B
c:\Users\bryan\Proyecto_EDA_UTEC\.venv\Lib\site-packages\sklearn\utils\_array_api.py:0: size=389 KiB, count=7, average=55.6 KiB
c:\Users\bryan\Proyecto_EDA_UTEC\src\bplus_tree.py:0: size=268 KiB, count=5817, average=47 B
c:\Users\bryan\Proyecto_EDA_UTEC\.venv\Lib\site-packages\pandas\core\internals\managers.py:0: size=170 KiB, count=108, average=1615 B
c:\Users\bryan\Proyecto_EDA_UTEC\src\pipeline.py:0: size=164 KiB, count=2555, average=66 B
c:\Users\bryan\Proyecto_EDA_UTEC\.venv\Lib\site-packages\pandas\core\arraylike.py:0: size=130 K

## 4. Diagnóstico de memoria: DataFrame vs lista de eventos

In [4]:

import sys
import gc
import psutil
import os
import pandas as pd

N = 100_000

process = psutil.Process(os.getpid())

gc.collect()
rss_before = process.memory_info().rss

# DataFrame de 100K eventos
df_memory = df.head(N).copy()

rss_dataframe = process.memory_info().rss

# Convertir a diccionarios
events_memory = df_memory.to_dict("records")

rss_dicts = process.memory_info().rss

print("=== COMPARACIÓN DE REPRESENTACIONES ===")

print(
    "Memoria profunda DataFrame:",
    f"{df_memory.memory_usage(deep=True).sum()/1024**2:.2f} MiB"
)

print(
    "RSS antes:",
    f"{rss_before/1024**2:.2f} MiB"
)

print(
    "RSS después DataFrame:",
    f"{rss_dataframe/1024**2:.2f} MiB"
)

print(
    "RSS después diccionarios:",
    f"{rss_dicts/1024**2:.2f} MiB"
)

print(
    "Incremento RSS al convertir:",
    f"{(rss_dicts-rss_dataframe)/1024**2:.2f} MiB"
)

# Liberar objetos temporales
del events_memory, df_memory
gc.collect()

print("\n¡Diagnóstico completado!")


=== COMPARACIÓN DE REPRESENTACIONES ===
Memoria profunda DataFrame: 16.38 MiB
RSS antes: 219.08 MiB
RSS después DataFrame: 225.16 MiB
RSS después diccionarios: 327.46 MiB
Incremento RSS al convertir: 102.30 MiB

¡Diagnóstico completado!


## 5. Lectura incremental de REES46

In [5]:

import pandas as pd
import time
import gc
import os
import psutil

CHUNK_SIZE = 10_000
MAX_EVENTS = 100_000

process = psutil.Process(os.getpid())

gc.collect()
rss_initial = process.memory_info().rss

total_events = 0
total_chunks = 0
max_chunk_memory = 0

start = time.perf_counter()

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    nrows=MAX_EVENTS
)

for chunk in reader:

    total_chunks += 1
    total_events += len(chunk)

    # Medir memoria del bloque actual
    chunk_memory = chunk.memory_usage(
        deep=True
    ).sum()

    max_chunk_memory = max(
        max_chunk_memory,
        chunk_memory
    )

    # Comprobar que se mantienen las 9 columnas
    assert len(chunk.columns) == 9

    # No acumular bloques ni convertir a diccionarios
    del chunk

elapsed = time.perf_counter() - start

gc.collect()
rss_final = process.memory_info().rss

print("=== LECTURA INCREMENTAL REES46 ===")
print(f"Eventos leídos: {total_events:,}")
print("Bloques procesados:", total_chunks)
print(f"Tamaño de bloque: {CHUNK_SIZE:,}")
print(
    f"Memoria máxima de un DataFrame de bloque: "
    f"{max_chunk_memory / 1024**2:.2f} MiB"
)
print(f"Tiempo de lectura: {elapsed:.4f} s")
print(
    f"Variación RSS: "
    f"{(rss_final-rss_initial)/1024**2:.2f} MiB"
)

assert total_events == MAX_EVENTS
assert total_chunks == 10

print("\n¡Lectura incremental validada!")


=== LECTURA INCREMENTAL REES46 ===
Eventos leídos: 100,000
Bloques procesados: 10
Tamaño de bloque: 10,000
Memoria máxima de un DataFrame de bloque: 1.49 MiB
Tiempo de lectura: 0.0581 s
Variación RSS: 0.38 MiB

¡Lectura incremental validada!


## 6. Validación de ingesta incremental vs implementación original

In [6]:

from src.streaming_ingestion import StreamingIngestion
from src.pipeline import run_ingestion_pipeline

import pandas as pd
import numpy as np


N = 10_000
CHUNK_SIZE = 2_000
BATCH_SIZE = 1_000


def iter_rees46(path, nrows, chunksize):
    """Genera eventos normalizados por bloques."""

    offset = 0

    reader = pd.read_csv(
        path,
        nrows=nrows,
        chunksize=chunksize
    )

    for chunk in reader:
        chunk["event_id"] = np.arange(
            offset,
            offset + len(chunk)
        )

        chunk["event_timestamp"] = pd.to_datetime(
            chunk["event_time"],
            utc=True
        )

        offset += len(chunk)

        yield from chunk.to_dict("records")


# Alternativa original
events_original = list(
    iter_rees46(DATA_PATH, N, CHUNK_SIZE)
)

original = run_ingestion_pipeline(
    events_original,
    batch_size=BATCH_SIZE,
    dedup_method="set"
)

# Alternativa incremental
streaming = StreamingIngestion(
    batch_size=BATCH_SIZE
)

# Consumir la salida y guardar SOLO los IDs
ids_streaming = [
    event["event_id"]
    for event in streaming.process(
        iter_rees46(DATA_PATH, N, CHUNK_SIZE)
    )
]

ids_original = [
    event["event_id"]
    for event in original["events"]
]

print("=== COMPARACIÓN DE INGESTA ===")
print("Eventos originales:", len(ids_original))
print("Eventos incrementales:", len(ids_streaming))
print("Orden idéntico:", ids_original == ids_streaming)

print("\n=== MÉTRICAS ===")
print("Original:", original["metrics"])
print("Incremental:", streaming.metrics)

# Comparar las métricas funcionales
for key in [
    "received", "valid", "invalid",
    "duplicates", "output"
]:
    assert original["metrics"][key] == streaming.metrics[key]

assert ids_original == ids_streaming

print("\n¡Ingesta incremental validada correctamente!")


=== COMPARACIÓN DE INGESTA ===
Eventos originales: 10000
Eventos incrementales: 10000
Orden idéntico: True

=== MÉTRICAS ===
Original: {'received': 10000, 'valid': 10000, 'invalid': 0, 'duplicates': 0, 'output': 10000, 'dedup_method': 'set'}
Incremental: {'received': 10000, 'valid': 10000, 'invalid': 0, 'duplicates': 0, 'output': 10000}

¡Ingesta incremental validada correctamente!


## 7. Validación de duplicados entre bloques

In [7]:

from src.streaming_ingestion import StreamingIngestion
from src.pipeline import run_ingestion_pipeline

# Tomar 12 eventos reales y normalizados
eventos_base = df.head(12).to_dict("records")

# Simular retransmisiones del primer, cuarto y octavo evento
# en posiciones posteriores.
eventos_prueba = eventos_base + [
    eventos_base[0].copy(),
    eventos_base[3].copy(),
    eventos_base[7].copy()
]

# Implementación original
original = run_ingestion_pipeline(
    eventos_prueba,
    batch_size=4,
    dedup_method="set"
)

# Implementación incremental
streaming = StreamingIngestion(batch_size=4)

stream_events = list(
    streaming.process(iter(eventos_prueba))
)

ids_original = [
    event["event_id"]
    for event in original["events"]
]

ids_streaming = [
    event["event_id"]
    for event in stream_events
]

print("=== RETRANSMISIONES ENTRE BLOQUES ===")
print("Eventos recibidos:", streaming.metrics["received"])
print("Duplicados detectados:", streaming.metrics["duplicates"])
print("Eventos finales:", streaming.metrics["output"])

print("\nIDs originales:", ids_original)
print("IDs streaming:", ids_streaming)

assert ids_original == ids_streaming == list(range(12))
assert streaming.metrics["duplicates"] == 3
assert streaming.metrics["output"] == 12

print("\n¡Deduplicación incremental validada!")


=== RETRANSMISIONES ENTRE BLOQUES ===
Eventos recibidos: 15
Duplicados detectados: 3
Eventos finales: 12

IDs originales: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
IDs streaming: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

¡Deduplicación incremental validada!


## 8. Benchmark de memoria de StreamingIngestion

In [8]:

import gc
import time
import psutil
import os

from src.streaming_ingestion import StreamingIngestion

N = 100_000
CHUNK_SIZE = 2_000
BATCH_SIZE = 1_000

process = psutil.Process(os.getpid())

gc.collect()
rss_before = process.memory_info().rss

streaming = StreamingIngestion(
    batch_size=BATCH_SIZE
)

start = time.perf_counter()
processed_count = 0

# No convertir la salida en una lista
for event in streaming.process(
    iter_rees46(DATA_PATH, N, CHUNK_SIZE)
):
    processed_count += 1

elapsed = time.perf_counter() - start
rss_after = process.memory_info().rss

print("=== BENCHMARK STREAMING ===")
print(f"Eventos procesados: {processed_count:,}")
print(f"Tiempo total: {elapsed:.4f} s")
print(f"Throughput: {processed_count / elapsed:,.0f} eventos/s")
print(f"RSS inicial: {rss_before / 1024**2:.2f} MiB")
print(f"RSS final: {rss_after / 1024**2:.2f} MiB")
print(
    f"Variación RSS: "
    f"{(rss_after - rss_before) / 1024**2:.2f} MiB"
)
print(
    f"IDs únicos almacenados: "
    f"{len(streaming.seen_ids):,}"
)
print("Métricas:", streaming.metrics)

assert processed_count == 100_000
assert streaming.metrics["output"] == processed_count

print("\n¡Benchmark incremental completado!")


=== BENCHMARK STREAMING ===
Eventos procesados: 100,000
Tiempo total: 1.0100 s
Throughput: 99,011 eventos/s
RSS inicial: 234.82 MiB
RSS final: 241.19 MiB
Variación RSS: 6.38 MiB
IDs únicos almacenados: 100,000
Métricas: {'received': 100000, 'valid': 100000, 'invalid': 0, 'duplicates': 0, 'output': 100000}

¡Benchmark incremental completado!


## 9. Benchmark repetido: comparación estadística

In [9]:

import pandas as pd

# Resultados de las 10 ejecuciones independientes
benchmark_data = {
    "metodo": ["original"] * 5 + ["streaming"] * 5,
    "tiempo_s": [
        1.1202, 1.0899, 1.1158, 1.1238, 1.1337,
        1.0285, 1.0108, 1.0303, 1.0640, 1.0117
    ],
    "throughput": [
        89271, 91749, 89624, 88983, 88211,
        97231, 98927, 97063, 93982, 98848
    ],
    "rss_incremento_mib": [
        112.43, 111.71, 112.12, 112.33, 112.41,
        16.12, 15.38, 16.12, 16.32, 16.39
    ]
}

df_benchmark = pd.DataFrame(benchmark_data)

summary = (
    df_benchmark
    .groupby("metodo")
    .agg(
        tiempo_mediana_s=("tiempo_s", "median"),
        throughput_mediana=("throughput", "median"),
        rss_mediana_mib=("rss_incremento_mib", "median")
    )
)

original = summary.loc["original"]
streaming = summary.loc["streaming"]

reduccion_memoria = (
    1 - streaming["rss_mediana_mib"]
    / original["rss_mediana_mib"]
) * 100

reduccion_tiempo = (
    1 - streaming["tiempo_mediana_s"]
    / original["tiempo_mediana_s"]
) * 100

print("=== RESUMEN ESTADÍSTICO ===")
display(summary.round(3))

print(f"Reducción memoria: {reduccion_memoria:.2f}%")
print(f"Reducción tiempo: {reduccion_tiempo:.2f}%")

results_dir = PROJECT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

df_benchmark.to_csv(
    results_dir / "benchmark_ingestion_raw.csv",
    index=False
)

summary.to_csv(
    results_dir / "benchmark_ingestion_summary.csv"
)

print("\n¡Resultados guardados correctamente!")


=== RESUMEN ESTADÍSTICO ===


,tiempo_mediana_s,throughput_mediana,rss_mediana_mib
metodo,,,
original,1.120,89271.0,112.33
streaming,1.028,97231.0,16.12


Reducción memoria: 85.65%
Reducción tiempo: 8.19%

¡Resultados guardados correctamente!
